In [127]:
#################################### IMPORTAR STOCK INICIAL ####################################
####################################                        ####################################

import pandas as pd
import glob
import os

ruta = r"C:\Users\Usuario\Downloads\MOVIMIENTOS DE STOCK\INVENTARIO PRUEBA"

# Obtener todos los archivos Excel, excluyendo los temporales
archivos = [
    a for a in glob.glob(os.path.join(ruta, "*.xlsx"))
    if not os.path.basename(a).startswith("~$")
]

lista_df = []

for archivo in archivos:
    # Leer todas las hojas del archivo (encabezado en la fila 2)
    hojas = pd.read_excel(archivo, sheet_name=None, header=1)

    for nombre_hoja, df_hoja in hojas.items():
        # Agregar información del origen
        df_hoja["Archivo"] = os.path.basename(archivo)
        df_hoja["Hoja"] = nombre_hoja

        lista_df.append(df_hoja)

# Unir todos los DataFrames
df = pd.concat(lista_df, ignore_index=True)

# Eliminar filas donde la columna LOCAL tenga el valor "LOCAL"
df = df[
    df["LOCAL"].astype(str).str.strip().str.upper() != "LOCAL"
].copy()

# (Opcional) Reiniciar el índice
df.reset_index(drop=True, inplace=True)

# Mostrar todas las columnas
pd.set_option("display.max_columns", None)


######################################  TRANSFORMAR ARCHIVOS DE STOCKS   #####################################

cols_base = ["LOCAL", "AREA", "CATEGORIA", "ARTICULO", "VALOR","UNIDAD"]

keyword = "STOCK MARTES"

cols_target = [c for c in df.columns if keyword in c]

cols_left = []
rename_map = {}

for c in cols_target:
    idx = df.columns.get_loc(c)
    
    if idx > 0:
        left_col = df.columns[idx - 5]
        cols_left.append(left_col)
        
        # renombrar la columna izquierda usando el nombre de la columna target
        rename_map[left_col] = f"FECHA {c}"

# construir columnas finales
cols_final = []
for c in cols_base + cols_left + cols_target:
    if c not in cols_final:
        cols_final.append(c)

df_stock_inicial = df[cols_final].rename(columns=rename_map)

# Buscar la columna que comienza con "FECHA STOCK"
col_fecha = [c for c in df_stock_inicial.columns if c.startswith("STOCK")][0]

# Renombrarla a "FECHA"
df_stock_inicial.rename(columns={col_fecha: "STOCK"}, inplace=True)

# Buscar la columna que comienza con "FECHA STOCK"
col_fecha = [c for c in df_stock_inicial.columns if c.startswith("FECHA STOCK")][0]

# Renombrarla a "FECHA"
df_stock_inicial.rename(columns={col_fecha: "FECHA"}, inplace=True)

# Extraer únicamente la fecha
df_stock_inicial["FECHA"] = (
    df_stock_inicial["FECHA"]
    .str.extract(r'(\d{2}/\d{2}/\d{4})', expand=False)
)

#df_stock_inicial["FECHA"] = (
#    df_stock_inicial["FECHA"]
#    .str.extract(r'(\d{2}/\d{2}/\d{4})')
#)

df_stock_inicial["FECHA"] = pd.to_datetime(
    df_stock_inicial["FECHA"],
    format="%d/%m/%Y"
)

# ==========================================
# RENOMBRAR COLUMNAS
# ==========================================

df_stock_inicial = df_stock_inicial.rename(columns={
    "ARTICULO": "PA (X1,X2)",
    "STOCK": "PAs/Bandejas",
})

# Crear la columna "DIA" con el nombre del día de la semana en mayúsculas
df_stock_inicial['DIA'] = df_stock_inicial['FECHA'].dt.day_name(locale='es_ES').str.upper()

df_stock_inicial["TIPO"] = "STOCK INICIAL"

df_stock_inicial = df_stock_inicial[['TIPO', 'PA (X1,X2)', 'FECHA', 'PAs/Bandejas', 'DIA',"UNIDAD"]]


# Columnas a conservar
columnas = ["TIPO","PA (X1,X2)","FECHA","PAs/Bandejas","UNIDAD"]
# Conservar solo esas columnas
df_stock_inicial = df_stock_inicial[columnas]


df_stock_inicial = df_stock_inicial.rename(columns={
    "fecha": "FECHA",
    "PA (X1,X2)": "ITEM",
    "PAs/Bandejas": "CANTIDAD",
    "Unidades": "UNIDAD",
    "TIPO": "TIPO_MOV_1",
    })

df_stock_inicial["NUM ORDEN"] = ""
df_stock_inicial["Referencia"] = ""
df_stock_inicial["PROVEEDOR"] = ""
df_stock_inicial["Comprador"] = ""
df_stock_inicial["Empresa"] = ""
df_stock_inicial["TIPO_MOV_2"] = "STOCK INICIAL"
df_stock_inicial["PRECIO"] = 2.0

df_stock_inicial["TOTAL"] = df_stock_inicial["CANTIDAD"] * df_stock_inicial["PRECIO"]

orden_columnas = ["TIPO_MOV_1","TIPO_MOV_2","NUM ORDEN","Referencia","Empresa","FECHA","PROVEEDOR","Comprador","ITEM","CANTIDAD","UNIDAD","PRECIO","TOTAL"]

df_stock_inicial = df_stock_inicial[orden_columnas]

#df_stock_inicial = df_stock_inicial[df_stock_inicial["CANTIDAD"] != 0]

#df_stock_inicial = df_stock_inicial[df_stock_inicial["ITEM"] == "HUEVOS"]

df_stock_inicial.head()

,TIPO_MOV_1,TIPO_MOV_2,NUM ORDEN,Referencia,Empresa,FECHA,PROVEEDOR,Comprador,ITEM,CANTIDAD,UNIDAD,PRECIO,TOTAL
0,STOCK INICIAL,STOCK INICIAL,,,,2026-07-14,,,TE A GRANEL,19,BOLSA X 100 GR,2.0,38.0
1,STOCK INICIAL,STOCK INICIAL,,,,2026-07-14,,,SAL DE MESA,119,KG,2.0,238.0
2,STOCK INICIAL,STOCK INICIAL,,,,2026-07-14,,,POLVO DE HORNEAR,60,SOBRE X 20 GR,2.0,120.0
3,STOCK INICIAL,STOCK INICIAL,,,,2026-07-14,,,PIMIENTA MOLIDA,17,BOLSA X 200 GR,2.0,34.0
4,STOCK INICIAL,STOCK INICIAL,,,,2026-07-14,,,PIMIENTA ENTERA,13,BOLSA X 200 GR,2.0,26.0


In [154]:
#################################### IMPORTAR STOCK FINAL ####################################
####################################                        ####################################

import pandas as pd
import glob
import os

ruta = r"C:\Users\Usuario\Downloads\MOVIMIENTOS DE STOCK\INVENTARIO PRUEBA"

# Obtener todos los archivos Excel, excluyendo los temporales
archivos = [
    a for a in glob.glob(os.path.join(ruta, "*.xlsx"))
    if not os.path.basename(a).startswith("~$")
]

lista_df = []

for archivo in archivos:
    # Leer todas las hojas del archivo (encabezado en la fila 2)
    hojas = pd.read_excel(archivo, sheet_name=None, header=1)

    for nombre_hoja, df_hoja in hojas.items():
        # Agregar información del origen
        df_hoja["Archivo"] = os.path.basename(archivo)
        df_hoja["Hoja"] = nombre_hoja

        lista_df.append(df_hoja)

# Unir todos los DataFrames
df = pd.concat(lista_df, ignore_index=True)

# Eliminar filas donde la columna LOCAL tenga el valor "LOCAL"
df = df[
    df["LOCAL"].astype(str).str.strip().str.upper() != "LOCAL"
].copy()

# (Opcional) Reiniciar el índice
df.reset_index(drop=True, inplace=True)

# Mostrar todas las columnas
pd.set_option("display.max_columns", None)


######################################  TRANSFORMAR ARCHIVOS DE STOCKS   #####################################

cols_base = ["LOCAL", "AREA", "CATEGORIA", "ARTICULO", "VALOR","UNIDAD"]

keyword = "STOCK MARTES"

cols_target = [c for c in df.columns if keyword in c]

cols_left = []
rename_map = {}

for c in cols_target:
    idx = df.columns.get_loc(c)
    
    if idx > 0:
        left_col = df.columns[idx - 5]
        cols_left.append(left_col)
        
        # renombrar la columna izquierda usando el nombre de la columna target
        rename_map[left_col] = f"FECHA {c}"

# construir columnas finales
cols_final = []
for c in cols_base + cols_left + cols_target:
    if c not in cols_final:
        cols_final.append(c)

df_stock_final = df[cols_final].rename(columns=rename_map)

# Buscar la columna que comienza con "FECHA STOCK"
col_fecha = [c for c in df_stock_final.columns if c.startswith("STOCK")][0]

# Renombrarla a "FECHA"
df_stock_final.rename(columns={col_fecha: "STOCK"}, inplace=True)

# Buscar la columna que comienza con "FECHA STOCK"
col_fecha = [c for c in df_stock_final.columns if c.startswith("FECHA STOCK")][0]

# Renombrarla a "FECHA"
df_stock_final.rename(columns={col_fecha: "FECHA"}, inplace=True)

# Extraer únicamente la fecha
df_stock_final["FECHA"] = (
    df_stock_final["FECHA"]
    .str.extract(r'(\d{2}/\d{2}/\d{4})', expand=False)
)

#df_stock_final["FECHA"] = (
#    df_stock_final["FECHA"]
#    .str.extract(r'(\d{2}/\d{2}/\d{4})')
#)

df_stock_final["FECHA"] = pd.to_datetime(
    df_stock_final["FECHA"],
    format="%d/%m/%Y"
)

# ==========================================
# RENOMBRAR COLUMNAS
# ==========================================

df_stock_final = df_stock_final.rename(columns={
    "ARTICULO": "PA (X1,X2)",
    "STOCK": "PAs/Bandejas",
})

# Crear la columna "DIA" con el nombre del día de la semana en mayúsculas
df_stock_final['DIA'] = df_stock_final['FECHA'].dt.day_name(locale='es_ES').str.upper()

df_stock_final["TIPO"] = "STOCK FINAL"

df_stock_final = df_stock_final[['TIPO', 'PA (X1,X2)', 'FECHA', 'PAs/Bandejas', 'DIA',"UNIDAD"]]


# Columnas a conservar
columnas = ["TIPO","PA (X1,X2)","FECHA","PAs/Bandejas","UNIDAD"]
# Conservar solo esas columnas
df_stock_final = df_stock_final[columnas]


df_stock_final = df_stock_final.rename(columns={
    "fecha": "FECHA",
    "PA (X1,X2)": "ITEM",
    "PAs/Bandejas": "CANTIDAD",
    "Unidades": "UNIDAD",
    "TIPO": "TIPO_MOV_1",
    })

df_stock_final["NUM ORDEN"] = ""
df_stock_final["Referencia"] = ""
df_stock_final["PROVEEDOR"] = ""
df_stock_final["Comprador"] = ""
df_stock_final["Empresa"] = ""
df_stock_final["TIPO_MOV_2"] = "STOCK FINAL"
df_stock_final["PRECIO"] = 2.0

df_stock_final["TOTAL"] = df_stock_final["CANTIDAD"] * df_stock_final["PRECIO"]

orden_columnas = ["TIPO_MOV_1","TIPO_MOV_2","NUM ORDEN","Referencia","Empresa","FECHA","PROVEEDOR","Comprador","ITEM","CANTIDAD","UNIDAD","PRECIO","TOTAL"]

df_stock_final = df_stock_final[orden_columnas]

#df_stock_final = df_stock_final[df_stock_final["CANTIDAD"] != 0]

#df_stock_final = df_stock_final[df_stock_final["ITEM"] == "HUEVOS"]

# Columnas a conservar
columnas = ["ITEM","CANTIDAD"]
# Conservar solo esas columnas
df_stock_final = df_stock_final[columnas]

df_stock_final = df_stock_final.rename(columns={"CANTIDAD": "CANTIDAD_FINAL"})

df_stock_final.head()

,ITEM,CANTIDAD_FINAL
0,TE A GRANEL,19
1,SAL DE MESA,119
2,POLVO DE HORNEAR,60
3,PIMIENTA MOLIDA,17
4,PIMIENTA ENTERA,13


In [99]:
#################################### RECEPCION DE PAS ####################################
####################################                  ####################################

import pandas as pd

# Ruta del archivo
ruta = r"G:\.shortcut-targets-by-id\1IA3IXT7GB3a_ijXF-P_6Ja_WLyIvLj9B\ALMACEN CD\5. FORMATO RECEPCIONES\5, FORMATO RECEPCIONES DE  P A S.xlsx"

df_recepciones = pd.read_excel(ruta,sheet_name="RECEPCION PAs")

# Columnas a conservar
columnas = ["fecha","hora","PA","cantidad","Unidades"]

# Conservar solo esas columnas
df_recepciones = df_recepciones[columnas]

df_recepciones = df_recepciones.rename(columns={
    "fecha": "FECHA",
    "PA": "ITEM",
    "cantidad": "CANTIDAD",
    "Unidades": "UNIDAD",
    
})

df_recepciones["NUM ORDEN"] = ""
df_recepciones["Referencia"] = ""
df_recepciones["UNIDAD"] = "PA"
df_recepciones["PROVEEDOR"] = ""
df_recepciones["Comprador"] = ""
df_recepciones["Empresa"] = ""
df_recepciones["TIPO_MOV_1"] = "RECEPCIONES PA"
df_recepciones["TIPO_MOV_2"] = "RECEPCIONES PA"
df_recepciones["PRECIO"] = 2.0

df_recepciones["TOTAL"] = df_recepciones["CANTIDAD"] * df_recepciones["PRECIO"]

orden_columnas = ["TIPO_MOV_1","TIPO_MOV_2","NUM ORDEN","Referencia","Empresa","FECHA","PROVEEDOR","Comprador","ITEM","CANTIDAD","UNIDAD","PRECIO","TOTAL"]

df_recepciones = df_recepciones[orden_columnas]

df_recepciones = df_recepciones[df_recepciones["CANTIDAD"] != 0]

df_recepciones.head()

C:\Users\Usuario\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,TIPO_MOV_1,TIPO_MOV_2,NUM ORDEN,Referencia,Empresa,FECHA,PROVEEDOR,Comprador,ITEM,CANTIDAD,UNIDAD,PRECIO,TOTAL
0,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-06,,,PA MAIZ TOSTADO,4.0,PA,2.0,8.0
1,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-06,,,PA SENCA DE RES LIMPIA,25.0,PA,2.0,50.0
2,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-07,,,PA HUESO MANZANO,41.0,PA,2.0,82.0
3,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-07,,,PA BASE ESTOFADO MENU,42.6,PA,2.0,85.2
4,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-07,,,PA CARNE DE ALMUERZO - COSTILLA DE RES,13.0,PA,2.0,26.0


In [138]:
#################################### RECEPCION DE PROVEEDORES ####################################
####################################                          ####################################

import pandas as pd

# Ruta del archivo
ruta = r"C:\Users\Usuario\Downloads\MOVIMIENTOS DE STOCK\Transferir (stock.picking)(10).xlsx"

# Leer el archivo Excel
df_recepciones_odoo = pd.read_excel(ruta)

# Columnas a conservar
columnas = ["Documento origen","Referencia","Fecha de traslado","Movimientos de stock/Producto","Movimientos de stock/Cantidad",
    "Movimientos de stock/Unidad","Producto/Proveedores/Proveedor","Empresa"]

# Leer únicamente las columnas de interés
df_recepciones_odoo = pd.read_excel(ruta, usecols=columnas)

df_recepciones_odoo = df_recepciones_odoo.rename(columns={
    "Documento origen": "NUM ORDEN",
    "Movimientos de stock/Producto": "ITEM",
    "Fecha de traslado": "FECHA",
    "Movimientos de stock/Cantidad": "CANTIDAD",
    "Movimientos de stock/Unidad": "UNIDAD",
    "Producto/Proveedores/Proveedor": "PROVEEDOR"
})

df_recepciones_odoo["TIPO_MOV_1"] = "RECEPCION DE PROVEEDOR"
df_recepciones_odoo["TIPO_MOV_2"] = "RECEPCION DE PROVEEDOR"
df_recepciones_odoo["Comprador"] = ""
df_recepciones_odoo["PRECIO"] = 2.0

df_recepciones_odoo["TOTAL"] = df_recepciones_odoo["CANTIDAD"] * df_recepciones_odoo["PRECIO"]

orden_columnas = ["TIPO_MOV_1","TIPO_MOV_2","NUM ORDEN","Referencia","Empresa","FECHA","PROVEEDOR","Comprador","ITEM","CANTIDAD","UNIDAD","PRECIO","TOTAL"]

df_recepciones_odoo = df_recepciones_odoo[orden_columnas]

df_recepciones_odoo = df_recepciones_odoo[df_recepciones_odoo["CANTIDAD"] != 0]

df_recepciones_odoo = df_recepciones_odoo[df_recepciones_odoo["ITEM"] == "HARINA BLANCA"]

df_recepciones_odoo.head()

,TIPO_MOV_1,TIPO_MOV_2,NUM ORDEN,Referencia,Empresa,FECHA,PROVEEDOR,Comprador,ITEM,CANTIDAD,UNIDAD,PRECIO,TOTAL
0,RECEPCION DE PROVEEDOR,RECEPCION DE PROVEEDOR,P00035,ALM/IN/00030,Planta,2026-05-14 13:00:33,COMERCIALIZADORA FAST PIG E.I.R.L.,,PANCETA DE CERDO,96.55,KG PANCETA,2.0,193.1
1,RECEPCION DE PROVEEDOR,RECEPCION DE PROVEEDOR,P00039,ALM/IN/00031,Planta,2026-05-14 13:26:19,COMERCIALIZADORA FAST PIG E.I.R.L.,,COSTILLA DE CERDO,122.25,KG,2.0,244.5
2,RECEPCION DE PROVEEDOR,RECEPCION DE PROVEEDOR,P00051,ALM/IN/00042,Planta,2026-05-16 15:21:43,PERCY,,KETCHUP BALDE,1.00,BALDE,2.0,2.0
3,RECEPCION DE PROVEEDOR,RECEPCION DE PROVEEDOR,NaN,NaN,NaN,NaT,NaN,,LENTEJAS,1.00,KG,2.0,2.0
4,RECEPCION DE PROVEEDOR,RECEPCION DE PROVEEDOR,NaN,NaN,NaN,NaT,NaN,,MAICENA,10.00,KG,2.0,20.0


In [103]:
#################################### TRANSFERENCIAS ####################################
####################################                ####################################

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

# Ruta del archivo
ruta = r"C:\Users\Usuario\Downloads\MOVIMIENTOS DE STOCK\Orden de compra (purchase.order)(27).xlsx"

# Leer el archivo Excel
df_transf = pd.read_excel(ruta)

# ==========================================
# COMPLETAR DATOS VACÍOS
# ==========================================

columnas = ["Referencia de la orden","Empresa","Comprador","Referencia de proveedor","Proveedor","Fecha de la orden","Entrega esperada","AREA","Tipo de Requerimiento"]

df_transf[columnas] = df_transf[columnas].ffill()

df_transf["TIPO_MOV_2"] = (df_transf["Tipo de Requerimiento"].astype(str) + "-" + df_transf["AREA"].astype(str))

# ==========================================
# FECHA Y CÓDIGO ÚNICO
# ==========================================

df_transf["Fecha de la orden"] = pd.to_datetime(df_transf["Fecha de la orden"], errors="coerce")

df_transf["Fecha_formato"] = df_transf["Fecha de la orden"].dt.strftime("%d%m%Y")

df_transf["Codigo unico de la orden"] = (
    df_transf["Referencia de la orden"].astype(str)
    + "-"
    + df_transf["Fecha_formato"].astype(str)
)

df_transf.drop(columns=["Fecha_formato"], inplace=True)

# ==========================================
# IDENTIFICAR ITEM PRINCIPAL
# ==========================================

df_transf["item"] = df_transf["Líneas de la orden/Descripción"].where(
    df_transf["Líneas de la orden/Cantidad"] != 0,
    np.nan
)

df_transf["item"] = df_transf["item"].ffill()

df_transf["Codigo unico de la orden 2"] = (
    df_transf["Codigo unico de la orden"].astype(str)
    + "-"
    + df_transf["item"].astype(str)
)

# ==========================================
# EXTRAER OBSERVACIONES
# ==========================================

df_observaciones = df_transf[
    ["Codigo unico de la orden 2", "Líneas de la orden/Descripción"]
].copy()

col = "Líneas de la orden/Descripción"

df_observaciones = df_observaciones[
    df_observaciones[col].str.contains(r"\(.*?\)", na=False)
]

df_observaciones[col] = df_observaciones[col].str.extract(r"\((.*?)\)", expand=False)

# ==========================================
# CONSERVAR SOLO PRODUCTOS
# ==========================================

df_transf = df_transf[df_transf["Líneas de la orden/Cantidad"] != 0].copy()

df_transf.drop(
    columns=[
        "Líneas de la orden/Descripción"
        
    ],
    inplace=True
)

# ==========================================
# UNIR OBSERVACIONES
# ==========================================

df_exports_order = df_transf.merge(
    df_observaciones,
    on="Codigo unico de la orden 2",
    how="left"
)

df_exports_order.drop(columns=["item", "Codigo unico de la orden 2"], inplace=True)

df_exports_order["Líneas de la orden/Descripción"] = (
    df_exports_order["Líneas de la orden/Descripción"].fillna("-")
)

# Columnas a conservar
columnas = ["Referencia de la orden","TIPO_MOV_2","Empresa","Comprador","Entrega esperada",
    "Líneas de la orden/Producto","Líneas de la orden/Unidad","Líneas de la orden/Cantidad"]

# Conservar solo esas columnas
df_transf = df_transf[columnas]

df_transf = df_transf.rename(columns={
    "Referencia de la orden": "NUM ORDEN",
    "Entrega esperada": "FECHA",
    "Líneas de la orden/Producto": "ITEM",
    "Líneas de la orden/Unidad": "UNIDAD",
    "Líneas de la orden/Cantidad": "CANTIDAD"
})

df_transf["Referencia"] = ""
df_transf["PROVEEDOR"] = ""
df_transf["PRECIO"] = 2.0
df_transf["TIPO_MOV_1"] = "REQUERIMIENTO"

df_transf["TOTAL"] = df_transf["CANTIDAD"] * df_transf["PRECIO"]

orden_columnas = ["TIPO_MOV_1","TIPO_MOV_2","NUM ORDEN","Referencia","Empresa","FECHA","PROVEEDOR","Comprador","ITEM","CANTIDAD","UNIDAD","PRECIO","TOTAL"]

df_transf = df_transf[orden_columnas]

df_transf = df_transf[df_transf["CANTIDAD"] != 0]

df_transf.head()

,TIPO_MOV_1,TIPO_MOV_2,NUM ORDEN,Referencia,Empresa,FECHA,PROVEEDOR,Comprador,ITEM,CANTIDAD,UNIDAD,PRECIO,TOTAL
0,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28,,Renzo Chacon,ACELGA,16.0,UND,2.0,32.0
1,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28,,Renzo Chacon,AJI AMARILLO,50.0,KG,2.0,100.0
2,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28,,Renzo Chacon,AJO PELADO,12.0,KG,2.0,24.0
3,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28,,Renzo Chacon,CALABAZA VERDE,100.0,KG,2.0,200.0
4,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28,,Renzo Chacon,CEBOLLA VERDE,4.0,MAZO CEBOLLA VERDE,2.0,8.0


In [144]:
###EXPORTO EL EXCEL CON TODOS LOS DATAFRAMES COMBINADOS

import os
import pandas as pd

df_total = pd.concat([df_stock_inicial,df_recepciones, df_recepciones_odoo, df_transf],ignore_index=True)

#df_total = df_total[df_total["ITEM"] == "PAPA CANCHAN"]

# Ruta de la carpeta
ruta = r"C:\Users\Usuario\Downloads\MOVIMIENTOS DE STOCK"
# Nombre del archivo
archivo = os.path.join(ruta, "df_total.xlsx")
# Exportar a Excel
df_total.to_excel(archivo, index=False)

df_total

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\Usuario\\Downloads\\MOVIMIENTOS DE STOCK\\df_total.xlsx'

In [146]:
### DEFINO UN RANGO DE FECHAS DE TODOS LOS "TIPO_MOV_2" PERO CONSIDERO TAMBIEN UN DIA ANTES DEL RANGO AL STOCK INICIAL

import pandas as pd

# Asegurar que la columna FECHA sea de tipo datetime
df_total["FECHA"] = pd.to_datetime(df_total["FECHA"], dayfirst=True)

# Fechas del filtro
fecha_inicio = pd.Timestamp("2026-07-20")
fecha_fin = pd.Timestamp("2026-07-29")

# Día anterior
fecha_stock = fecha_inicio - pd.Timedelta(days=1)

# Filtro
df_filtrado = df_total[
    (
        (df_total["FECHA"] >= fecha_inicio) &
        (df_total["FECHA"] <= fecha_fin)
    )
    |
    (
        (df_total["FECHA"] == fecha_stock) &
        (df_total["TIPO_MOV_2"] == "STOCK INICIAL")
    )
]
df_filtrado

,TIPO_MOV_1,TIPO_MOV_2,NUM ORDEN,Referencia,Empresa,FECHA,PROVEEDOR,Comprador,ITEM,CANTIDAD,UNIDAD,PRECIO,TOTAL
634,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-20 00:00:00,,,PA LLATAN,30.0,PA,2.0,60.0
635,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-20 00:00:00,,,PA SALSA DE POLLO,7.0,PA,2.0,14.0
636,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-20 00:00:00,,,PA FILETE DE POLLO,62.0,PA,2.0,124.0
637,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-20 00:00:00,,,PA SENCA DE RES LIMPIA,26.5,PA,2.0,53.0
638,RECEPCIONES PA,RECEPCIONES PA,,,,2026-07-21 00:00:00,,,PA HUESO MANZANO,51.0,PA,2.0,102.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
765,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28 00:00:00,,Renzo Chacon,ZANAHORIA,15.0,KG,2.0,30.0
766,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00830,,Planta,2026-07-28 00:00:00,,Renzo Chacon,PEREJIL,2.0,MAZO,2.0,4.0
767,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00829,,Planta,2026-07-28 17:17:48,,Renzo Chacon,AGUJA DE RES,7.5,KG,2.0,15.0
768,REQUERIMIENTO,PEDIDO PROVEEDOR-COCINA,P00828,,Planta,2026-07-28 17:12:26,,Renzo Chacon,POLLO ENTERO TROZADO,65.0,KG POLLO TROZADO,2.0,130.0


In [148]:
### Punto de partido para la validacion del stock deseable con el stock final real

df_resumen = (
    df_filtrado
    .groupby(['ITEM', 'UNIDAD'], as_index=False)['CANTIDAD']
    .sum()
)
df_resumen.head()

,ITEM,UNIDAD,CANTIDAD
0,ACELGA,UND,16.0
1,AGUJA DE RES,KG,7.5
2,AJI AMARILLO,KG,50.0
3,AJO PELADO,KG,12.0
4,CALABAZA VERDE,KG,100.0


In [160]:
df_merge = df_resumen.merge(df_stock_final,on='ITEM',how='left')

df_merge['DIFERENCIA'] = df_merge['CANTIDAD_FINAL'] - df_merge['CANTIDAD']

df_merge['ESTADO'] = np.select([df_merge['DIFERENCIA'] < 0, df_merge['DIFERENCIA'] == 0, df_merge['DIFERENCIA'] > 0],['FALTA','OK','EXCEDE'])

df_merge.head()

,ITEM,UNIDAD,CANTIDAD,CANTIDAD_FINAL,DIFERENCIA,ESTADO
0,ACELGA,UND,16.0,0,-16.0,FALTA
1,AGUJA DE RES,KG,7.5,0,-7.5,FALTA
2,AJI AMARILLO,KG,50.0,0,-50.0,FALTA
3,AJO PELADO,KG,12.0,0,-12.0,FALTA
4,CALABAZA VERDE,KG,100.0,0,-100.0,FALTA
